# Start here: inspect the reconstruction dataset

The shortest route to reproduce the central result and check a concrete predicted mass spectrum.

**Data:** the frozen 3000-run production set (`data/raw`). **Acceptance:** all five normalized residual magnitudes strictly below one. These are *numerical widths*, not independently measured Gaussian standard deviations.

> Start from the repository root or its `notebooks/` folder. No recomputation of the 3000 DE optimizations is required.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Works in Jupyter, VS Code, and Colab if the repository is cloned locally.
workdir = Path.cwd().resolve()
candidates = [workdir, workdir.parent]
ROOT = next((p for p in candidates if (p / "data/raw/runs.csv").is_file()
             and (p / "src/model.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError(
        "Cannot find the repository root. Open the project folder (with data/ and src/) "
        "or clone the repository in the Colab session before running the notebook."
    )
sys.path.insert(0, str(ROOT / "src"))
from model import PARAMETERS, V_EW, MASS_WIDTHS, mass_matrix, spectrum
raw = pd.read_csv(ROOT / "data/raw/runs.csv")
print(f"Repository: {ROOT.name}; records: {len(raw):,}; v = {V_EW} GeV")

Repository: DE_Particle_paper_public; records: 3,000; v = 246.22 GeV


## Dataset completeness\nEach *target* contributes three optimization runs. The flags stored by the original optimizer are historical; the strict final criterion is recomputed from residuals.

In [2]:
residual_cols = ["z_H1", "z_H2", "z_H3", "z_charged", "z_kappa"]
z_abs = np.abs(raw[residual_cols].to_numpy(float))
mask_mass = (z_abs[:, :3] < 1).all(axis=1)
mask_charged = z_abs[:, 3] < 1
mask_kappa = z_abs[:, 4] < 1
mask_all = (z_abs < 1).all(axis=1)
accepted = raw.loc[mask_all].copy()
representative = (accepted.sort_values(["target_index", "loss", "restart"])
                  .drop_duplicates("target_index").sort_values("target_index"))
assert len(raw) == 3000 and len(accepted) == 2971 and len(representative) == 1000

summary = pd.DataFrame({"metric": ["Unique targets", "Optimization runs", "Mass pass (<1)", "Charged pass (<1)", "Kappa pass (<1)", "All five pass (<1)"],
                        "count": [raw.target_index.nunique(),len(raw),int(mask_mass.sum()),int(mask_charged.sum()),int(mask_kappa.sum()),int(mask_all.sum())]})
print(summary.to_string(index=False))
assert raw[['target_index','restart']].drop_duplicates().shape[0] == 3000
assert (raw.groupby('target_index').size()==3).all()

            metric  count
    Unique targets   1000
 Optimization runs   3000
    Mass pass (<1)   2977
 Charged pass (<1)   2994
   Kappa pass (<1)   3000
All five pass (<1)   2971


## One reproducible inverse image\nSelect target `0`, restart `2` (base seed 37). Recalculate the CP-even squared-mass matrix and diagonalize independently.

In [3]:
row = raw.loc[(raw.target_index == 0) & (raw.restart == 2)].iloc[0]
x = row[list(PARAMETERS)].to_numpy(float)
M2 = mass_matrix(x)
masses, R, kappa, charged, min_eig = spectrum(x)
mat = pd.DataFrame(M2,index=["rho1","rho2","xi"],columns=["rho1","rho2","xi"])
print("CP-even mass-squared matrix (GeV²):")
print(mat.round(6).to_string())
print("Target masses (GeV):",np.round(row[["target_H1","target_H2","target_H3"]].to_numpy(float),6))
print("Predicted masses (GeV):",np.round(masses,6))
print("Maximum normalized mass residual:",np.max(np.abs((masses-row[["target_H1","target_H2","target_H3"]].to_numpy(float))/MASS_WIDTHS)))
print("Charged scalar (GeV):",charged,"; kappa_2V:",kappa)
assert np.allclose(R.T @ M2 @ R,np.diag(masses*masses),atol=1e-8)

CP-even mass-squared matrix (GeV²):
             rho1          rho2            xi
rho1  9083.528099    -24.935362     90.290455
rho2   -24.935362  15893.870672  -1321.016639
xi      90.290455  -1321.016639  22645.014242
Target masses (GeV): [ 95.304188 125.078362 151.310701]
Predicted masses (GeV): [ 95.304187 125.078364 151.3107  ]
Maximum normalized mass residual: 2.5329561879061657e-05
Charged scalar (GeV): 129.7347260859375 ; kappa_2V: 0.9826481755258525


## Next\nContinue with [`01_selection_audit.ipynb`](01_selection_audit.ipynb), then [`02_numerical_preimages.ipynb`](02_numerical_preimages.ipynb). All notebook outputs are generated from the released frozen CSVs.